In [2]:
import sys
import pandas as pd
import time

# 1. 모듈 임포트 및 경로 설정
sys.path.append(r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\collect")

from sec_data_pipeline.collectors.get_us_ticker import get_filtered_us_tickers
from sec_data_pipeline.collectors.rate_limiter import AdaptiveRateLimiter # 반드시 확인
from sec_data_pipeline.valuation.integrated_financial_analyzer_mysql_fixed import IntegratedFinancialAnalyzer
from sec_data_pipeline.storage.db_manager import DBManager
from DATA.stock_invest_function import get_db_host

# ---------------------------------------------------------
# [설정 항목] 여기서 날짜와 개수를 조절하세요
# ---------------------------------------------------------
START_DATE = "2025-01-01"  # ✅ 이 날짜 이후의 데이터만 수집/저장
MAX_TICKERS = 5000         # ✅ 테스트로 몇 개만 할지 결정 (전체는 None 또는 큰 숫자)
OFFSET = 0              # ✅ 시작 위치 (4000번 인덱스부터)
# ---------------------------------------------------------

# 2. DB 및 객체 초기화
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
db_manager = DBManager(db_info)
analyzer = IntegratedFinancialAnalyzer(db_info=db_info, wrds_conn_str="여기에_WRDS_주소")

# ✅ [중요] NameError 방지를 위한 rate_limiter 정의
rate_limiter = AdaptiveRateLimiter(
    max_calls=8,
    time_window=1.0,
    min_calls=3,
    backoff_factor=0.8,
)

headers = {"User-Agent": "Hoyoung Research <stox1224@gmail.com>"}

# 3. 티커 리스트 준비 (슬라이싱 적용)
ALL_TICKERS = get_filtered_us_tickers()
TICKER_LIST = ALL_TICKERS[OFFSET : OFFSET + MAX_TICKERS]

TICKER_LIST = ['A']

print(f"총 {len(TICKER_LIST)}개의 티커를 수집합니다. (기준일: {START_DATE})")

# 4. 메인 루프
for i, ticker in enumerate(TICKER_LIST, start=1):
    print(f"\n=== [{i}/{len(TICKER_LIST)}] {ticker} 처리 시작 ===")

    # 이제 정의된 rate_limiter를 사용하므로 에러가 나지 않습니다.
    rate_limiter.wait_if_needed()

    try:
        result = analyzer.analyze(ticker, headers=headers, table_name="us_fundq")
        if result is None:
            continue

        final_df, cik, entity_name = result

        # ✅ [추가] 날짜 필터링 로직 적용
        if not final_df.empty:
            final_df.index = pd.to_datetime(final_df.index)
            # 설정한 START_DATE 이후 데이터만 필터링
            final_df = final_df[final_df.index >= pd.to_datetime(START_DATE)]

        if final_df.empty:
            print(f"⚠ {ticker}: {START_DATE} 이후의 새로운 데이터가 없어 저장을 건너뜁니다.")
            continue

        # DB 저장용 인덱스 정리
        if final_df.index.name != "date":
            final_df.index.name = "date"

        db_manager.save_normalized_data(
            ticker=ticker,
            cik=cik,
            df=final_df,
            item_mapping=None
        )
        print(f"✓ {ticker} ({entity_name}) {len(final_df)}건 저장 완료")

    except Exception as e:
        msg = str(e)
        if "429" in msg:
            print(f"⚠ {ticker}: SEC 429 감지 → 60초 대기")
            rate_limiter.on_rate_limit_error()
            time.sleep(60)
            continue
        print(f"✗ {ticker} 처리 중 오류: {e}")
        continue

print("\n=== 작업 완료 ===")



✓ DB connected: 192.168.0.230:3307/investar


100%|██████████| 299/299 [00:00<00:00, 1005.94it/s]


제외된 기업 수: 1816
남은 기업 수: 4988
티커 수: 4988
총 1개의 티커를 수집합니다. (기준일: 2025-01-01)

=== [1/1] A 처리 시작 ===
[A] 통합 재무분석 시작
✓ Entity: AGILENT TECHNOLOGIES, INC. (CIK: 1090872)
✓ EDGAR 정규화 DF: 71 rows × 31 cols
✓ MySQL 연결 성공: 192.168.0.230:3307/investar
⚠ WRDS 쿼리 실패: Execution failed on sql 'SELECT * FROM us_fundq WHERE tic=%s': (1146, "Table 'investar.us_fundq' doesn't exist")
WRDS 데이터 없음 → EDGAR 그대로 반환
✓ EDGAR+WRDS 병합 DF: 71 rows × 31 cols
재무비율 계산 중...
  - 수익성 비율 계산...
  - 레버리지 비율 계산...
  - 유동성 비율 계산...
  - 효율성 비율 계산...
재무비율 계산 완료!
✓ 최종 DF (비율 포함): 71 rows × 47 cols
[A] 통합 재무분석 종료
✓ Saved A data: 3 dates x 47 items
✓ A (AGILENT TECHNOLOGIES, INC.) 3건 저장 완료

=== 작업 완료 ===


In [8]:
import pandas as pd
import pymysql
from typing import Dict, Optional, List


def quarter_end_from_report_date(report_date: pd.Timestamp) -> pd.Timestamp:
    y = report_date.year
    m = report_date.month

    if m in [1, 2, 3]:
        return pd.Timestamp(f"{y}-03-31")
    elif m in [4, 5, 6]:
        return pd.Timestamp(f"{y}-06-30")
    elif m in [7, 8, 9]:
        return pd.Timestamp(f"{y}-09-30")
    else:
        return pd.Timestamp(f"{y}-12-31")


import pandas as pd
from sqlalchemy import create_engine, text

def fetch_sec_financial_pivot_fixed(
    db_info: dict,
    ticker: str,
    table_name: str = "sec_financial_data",
    start_date: str = None,
    end_date: str = None,
    item_list: list = None,
):
    """
    SEC 재무데이터를 DB에서 읽어 period_end 기준으로 pivot(wide) 변환하여 반환.
    - index: period_end
    - columns: item_name (또는 item_list로 제한)
    - values: value
    """

    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
    )

    where = ["ticker = :ticker"]
    params = {"ticker": ticker}

    # ✅ DB 컬럼명이 현재 'date'라면 그게 period_end임 (현 구조 유지)
    # 나중에 period_end 컬럼을 만들었다면 아래 date를 period_end로 바꾸면 됨.
    date_col = "date"

    if start_date:
        where.append(f"{date_col} >= :start_date")
        params["start_date"] = start_date
    if end_date:
        where.append(f"{date_col} <= :end_date")
        params["end_date"] = end_date

    if item_list and len(item_list) > 0:
        # IN 절 파라미터 바인딩
        placeholders = []
        for i, it in enumerate(item_list):
            key = f"it{i}"
            placeholders.append(f":{key}")
            params[key] = it
        where.append(f"item_name IN ({', '.join(placeholders)})")

    sql = f"""
        SELECT
            {date_col} AS period_end,
            ticker,
            item_name,
            value
        FROM {table_name}
        WHERE {" AND ".join(where)}
        ORDER BY {date_col}, item_name
    """

    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn, params=params)

    if df.empty:
        return pd.DataFrame()

    # 타입 정리
    df["period_end"] = pd.to_datetime(df["period_end"], errors="coerce")
    df = df.dropna(subset=["period_end"])

    # ✅ pivot (wide)
    pivot = df.pivot_table(
        index="period_end",
        columns="item_name",
        values="value",
        aggfunc="last"
    ).sort_index()

    pivot.index.name = "period_end"
    return pivot



In [9]:
# 1) AAPL 전체 기간, 모든 item_name
fs_df = fetch_sec_financial_pivot_fixed(db_info, "A")
fs_df

item_name,accounts_payable,accounts_receivable,accrued_liabilities,accumulated_depreciation,asset_turnover,capital_expenditures,cash,cost_of_revenue,current_assets,current_liabilities,...,receivables_turnover,research_development,revenue,roa,roe,roic,short_term_debt,stockholders_equity,total_assets,total_liabilities
period_end,,,,,,,,,,,,,,,,,,,,,
2006-10-31,NaN,NaN,NaN,NaN,NaN,NaN,2.262000e+09,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.648000e+09,NaN,NaN
2007-10-31,NaN,NaN,NaN,NaN,NaN,NaN,1.826000e+09,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.234000e+09,NaN,NaN
2008-07-31,NaN,NaN,NaN,NaN,NaN,NaN,1.640000e+09,641000000.0,NaN,NaN,...,NaN,170000000.0,NaN,NaN,NaN,NaN,NaN,3.234000e+09,NaN,NaN
2008-10-31,308000000.0,7.700000e+08,409000000.0,NaN,NaN,NaN,1.405000e+09,641000000.0,3.182000e+09,1.330000e+09,...,NaN,170000000.0,NaN,NaN,NaN,4.851270,NaN,2.559000e+09,7.007000e+09,4.448000e+09
2009-01-31,308000000.0,7.700000e+08,409000000.0,NaN,NaN,34000000.0,1.362000e+09,577000000.0,3.182000e+09,1.330000e+09,...,NaN,169000000.0,NaN,NaN,NaN,0.534084,NaN,2.559000e+09,7.007000e+09,4.448000e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31,497000000.0,1.227000e+09,309000000.0,1.223000e+09,0.145632,90000000.0,1.779000e+09,542000000.0,4.256000e+09,2.389000e+09,...,1.22993,127000000.0,1.578000e+09,9.68114,18.3056,3.321140,795000000.0,5.903000e+09,1.099600e+10,5.093000e+09
2024-10-31,540000000.0,1.324000e+09,368000000.0,1.304000e+09,0.139590,90000000.0,1.329000e+09,542000000.0,3.959000e+09,1.895000e+09,...,1.20688,127000000.0,1.578000e+09,10.79220,20.7783,3.183660,45000000.0,5.898000e+09,1.184600e+10,5.948000e+09
2025-01-31,547000000.0,1.328000e+09,258000000.0,1.304000e+09,0.147056,97000000.0,1.467000e+09,542000000.0,4.107000e+09,1.869000e+09,...,1.28174,113000000.0,1.681000e+09,10.41030,19.4842,3.732370,16000000.0,6.027000e+09,1.191400e+10,5.887000e+09
